# `Quantity` & Units — demo

Step 1 of the framework: every value is a unit-bearing `Quantity` that can carry per-scenario and per-mode variation, with worst-case interval math.

This notebook walks through:

1. The shared Pint registry (`framework.units`).
2. `Constant` — scenario/mode-invariant values.
3. `RangeQuantity` — worst-case `(lo, hi)` with interval arithmetic.
4. **Scenarios** — named corners (`cold_low_vin`, `nominal`, `hot_high_vin`, …).
5. **Modes** — system-wide operating states (`sleep`, `active`, …).
6. Combined modes × scenarios.
7. A worked voltage-divider example.
8. The `within()` predicate — worst-case spec checking.

In [1]:
from framework import Constant, Quantity, RangeQuantity, units
from framework.units import V, mV, A, mA, uA, Ohm, kOhm, W, mW

## 1. The shared Pint registry

`framework.units` exposes a single shared `pint.UnitRegistry`. Always import unit symbols from here — mixing registries breaks unit interoperability.

In [2]:
v = 3.3 * V
i = 2 * mA
print(f"V = {v}")
print(f"I = {i}")
print(f"P = V*I = {(v * i).to(mW):~P}")
print(f"R = V/I = {(v / i).to(kOhm):~P}")

V = 3.3 V
I = 2 mA
P = V*I = 6.6 mW
R = V/I = 1.65 kΩ


In [3]:
# Dimensional mismatch raises at compute time:
import pint
try:
    (1 * V) + (1 * A)
except pint.DimensionalityError as e:
    print(f"caught: {e}")

caught: Cannot convert from 'volt' ([mass] * [length] ** 2 / [time] ** 3 / [current]) to 'ampere' ([current])


## 2. `Constant` — scenario/mode-invariant values

Use `Constant(value, unit)` (or `Constant(value_with_pint_unit)`) for things that don't vary.

In [4]:
vcc = Constant(3.3, V)
print(f"vcc          = {vcc.at()} {vcc.unit}")
print(f"vcc in mV    = {vcc.to(mV).at()} {vcc.to(mV).unit}")
print(f"vcc * 2      = {(vcc * 2).at()} {(vcc * 2).unit}")
print(f"vcc + 0.1 V  = {(vcc + Constant(0.1, V)).at()} V")

vcc          = 3.3 V
vcc in mV    = 3300.0 mV
vcc * 2      = 6.6 V²
vcc + 0.1 V  = 3.4 V


## 3. `RangeQuantity` — worst-case intervals

For values you only know within a min/max range. Arithmetic propagates the interval:

- `[a, b] + [c, d]` → `[a+c, b+d]`
- `[a, b] - [c, d]` → `[a-d, b-c]` (widens)
- `[a, b] * [c, d]` → `[min, max]` of the four corner products
- `[a, b] / [c, d]` → same, **raises** if divisor straddles zero

In [5]:
vbat = RangeQuantity(9.0, 16.0, V)  # automotive 12V system, worst-case envelope
print(f"VBAT raw range:      {vbat.at()} V")

drop = RangeQuantity(0.3, 0.7, V)   # reverse-polarity diode drop
vsys = vbat - drop
print(f"VBAT - diode drop:   {vsys.at()} V   <- subtraction widens the range")

VBAT raw range:      (9.0, 16.0) V
VBAT - diode drop:   (8.3, 15.7) V   <- subtraction widens the range


In [6]:
# I^2*R dissipation over a current and resistance range:
i_range = RangeQuantity(1.0, 2.0, A)
r_rds   = RangeQuantity(20e-3, 40e-3, Ohm)  # MOSFET R_DS(on) over temperature
p = i_range * i_range * r_rds
print(f"Dissipation worst-case: {p.to(mW).at()} mW")

Dissipation worst-case: (20.0, 160.0) mW


## 4. Scenarios — named corners

A Quantity can vary by **scenario** (e.g. `cold_low_vin`, `hot_high_vin`). Arithmetic propagates scenario-wise — same-named keys combine, so each scenario remains a self-consistent worldview.

In [7]:
vbat = Quantity(unit=V, by_scenario={
    "cold_low_vin":  9.0,
    "nominal":      12.0,
    "hot_high_vin": 16.0,
})

for s in ("cold_low_vin", "nominal", "hot_high_vin"):
    print(f"  vbat @ {s:14s} = {vbat.at(scenario=s):5.1f} V")

  vbat @ cold_low_vin   =   9.0 V
  vbat @ nominal        =  12.0 V
  vbat @ hot_high_vin   =  16.0 V


In [8]:
# Ohm's law per scenario:
r_load = Constant(100.0, Ohm)
i_load = (vbat / r_load).to(mA)

for s in ("cold_low_vin", "nominal", "hot_high_vin"):
    print(f"  i_load @ {s:14s} = {i_load.at(scenario=s):5.1f} mA")

  i_load @ cold_low_vin   =  90.0 mA
  i_load @ nominal        = 120.0 mA
  i_load @ hot_high_vin   = 160.0 mA


## 5. Modes — system-wide operating states

`by_mode` is the **outer** axis. The whole board is in one mode at a time (`sleep`, `active`, `diagnostic`, …). Children must share the parent's unit.

In [9]:
block_a_draw = Quantity(unit=A, by_mode={
    "sleep":      Constant(15e-6, A),
    "active":     Constant(0.15,  A),
    "diagnostic": Constant(0.28,  A),
})

for m in ("sleep", "active", "diagnostic"):
    val = block_a_draw.at(mode=m)
    if val < 1e-3:
        print(f"  {m:11s} {val*1e6:6.1f} uA")
    else:
        print(f"  {m:11s} {val*1e3:6.1f} mA")

  sleep         15.0 uA
  active       150.0 mA
  diagnostic   280.0 mA


## 6. Combined modes × scenarios

Mode is outer, scenario is inner. This is how a real block declares its current draw across the full state space.

In [10]:
# Quoting the design doc's Mike/Dan example, simplified:
block_a_3v3_draw = Quantity(unit=A, by_mode={
    "sleep":      Quantity(unit=A, by_scenario={"cold":  5e-6, "hot":  12e-6}),
    "active":     Quantity(unit=A, by_scenario={"cold": 80e-3, "hot": 220e-3}),
    "diagnostic": Quantity(unit=A, by_scenario={"cold":100e-3, "hot": 260e-3}),
})

print(f"{'mode':12s} {'cold':>10s} {'hot':>10s}")
for m in ("sleep", "active", "diagnostic"):
    c = block_a_3v3_draw.at(mode=m, scenario="cold")
    h = block_a_3v3_draw.at(mode=m, scenario="hot")
    print(f"{m:12s} {c*1e3:9.3f}mA {h*1e3:9.3f}mA")

mode               cold        hot
sleep            0.005mA     0.012mA
active          80.000mA   220.000mA
diagnostic     100.000mA   260.000mA


In [11]:
# Aggregating loads from three blocks (mode-axis preserved through addition):
block_b = Quantity(unit=A, by_mode={
    "sleep":      Constant(8e-6,  A),
    "active":     Constant(35e-3, A),
    "diagnostic": Constant(35e-3, A),
})
block_c = Quantity(unit=A, by_mode={
    "sleep":      Constant(3e-6,  A),
    "active":     Constant(12e-3, A),
    "diagnostic": Constant(20e-3, A),
})

# block_a_3v3_draw has scenarios; block_b/c don't — they broadcast.
total = block_a_3v3_draw + block_b + block_c

print(f"{'mode':12s} {'cold':>12s} {'hot':>12s}")
for m in ("sleep", "active", "diagnostic"):
    c = total.at(mode=m, scenario="cold")
    h = total.at(mode=m, scenario="hot")
    print(f"{m:12s} {c*1e3:11.3f}mA {h*1e3:11.3f}mA")

mode                 cold          hot
sleep              0.016mA       0.023mA
active           127.000mA     267.000mA
diagnostic       155.000mA     315.000mA


## 7. Worked example — VBAT divider into a 3V3 ADC

Top R = 10k, bottom R = 3.3k, ADC input range 0–3.3 V.
VBAT covers the automotive cold-crank / hot-high-vin envelope (9–16 V).

In [12]:
r_top = Constant(10.0, kOhm)
r_bot = Constant(3.3,  kOhm)
vbat  = Quantity(unit=V, by_scenario={
    "cold_low_vin":  9.0,
    "nominal":      12.0,
    "hot_high_vin": 16.0,
})

v_adc = vbat * r_bot / (r_top + r_bot)

print(f"{'scenario':14s} {'VBAT':>6s} {'V_ADC':>8s}")
for s in ("cold_low_vin", "nominal", "hot_high_vin"):
    print(f"{s:14s} {vbat.at(scenario=s):5.1f}V  {v_adc.at(scenario=s):7.3f}V")

print()
in_spec = v_adc.within(0 * V, 3.3 * V)
print(f"V_ADC stays inside [0, 3.3] V across all scenarios? {in_spec}")

scenario         VBAT    V_ADC
cold_low_vin     9.0V    2.233V
nominal         12.0V    2.977V
hot_high_vin    16.0V    3.970V

V_ADC stays inside [0, 3.3] V across all scenarios? False


## 8. `within()` — worst-case spec checking

Returns `True` iff **every** scenario/mode evaluation is inside the spec. This is the building block for the auto-generated contract-consistency checks the design doc describes in section 6.7.

In [13]:
vrail_good = Quantity(unit=V, by_scenario={"hot": 3.05, "cold": 3.55})
vrail_bad  = Quantity(unit=V, by_scenario={"hot": 2.95, "cold": 3.55})

spec_lo, spec_hi = 3.0 * V, 3.6 * V
print(f"good rail within [3.0, 3.6] V?  {vrail_good.within(spec_lo, spec_hi)}")
print(f"bad rail  within [3.0, 3.6] V?  {vrail_bad.within(spec_lo, spec_hi)}")

good rail within [3.0, 3.6] V?  True
bad rail  within [3.0, 3.6] V?  False


---

That's everything step 1 ships. The next implementation steps wire scenarios and modes into project config loaders (step 2), then bring Hamilton in as the DAG engine (step 4) so these `Quantity` values flow through real block-level analysis.